In [3]:
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Fetch Live 2025-26 Premier League Data
url = "https://fantasy.premierleague.com/api/bootstrap-static/"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
data = resp.json()

df_live = pd.DataFrame(data['elements'])
teams_dict = {t['id']: t['name'] for t in data['teams']}
pos_dict = {p['id']: p['singular_name_short'] for p in data['element_types']}

df_live['team_name'] = df_live['team'].map(teams_dict)
df_live['element_type'] = df_live['element_type'].map(pos_dict)
df_live['Player'] = df_live['first_name'] + " " + df_live['second_name']
df_live['Cost_mil'] = df_live['now_cost'] / 10.0

# Filter active players
df_model_ready = df_live[df_live['status'] != 'u'].copy()

# 2. Dynamic Threshold for Current Season Progress
# Using median minutes among active players to ensure both classes exist (0 and 1)
median_mins = df_model_ready[df_model_ready['minutes'] > 0]['minutes'].median()
print(f"Current Season Median Minutes: {median_mins}")

# 1 = High Absence (below median play-time), 0 = Durable / First-choice
df_model_ready['high_absence'] = (df_model_ready['minutes'] < median_mins).astype(int)

features = ['minutes', 'bps', 'now_cost']
X = df_model_ready[features].fillna(0)
y = df_model_ready['high_absence']

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 3. Balanced Random Forest Training
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=6,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train, y_train)

# 4. Safe Probability Extraction
probabilities = rf_model.predict_proba(X)
if probabilities.shape[1] > 1:
    df_model_ready['Risk_Prob'] = np.round(probabilities[:, 1] * 100, 1)
else:
    df_model_ready['Risk_Prob'] = 0.0

def map_verdict(p):
    if p >= 65: return '⛔ Red Flag: High Risk'
    elif p >= 40: return '⚠️ Amber: Moderate Workload'
    else: return '✅ Green: Durable Target'

df_model_ready['Verdict'] = df_model_ready['Risk_Prob'].apply(map_verdict)

# 5. Filter Targets strictly for External & Budget calibers (< 6.0m)
budget_targets = df_model_ready[
    (df_model_ready['team_name'] != 'Liverpool') &
    (df_model_ready['now_cost'] < 60) &
    (df_model_ready['now_cost'] >= 45)
].sort_values(by='Risk_Prob')

print(f"\nTotal evaluated external budget players: {len(budget_targets)}\n")
cols_to_show = ['Player', 'team_name', 'element_type', 'Cost_mil', 'minutes', 'Risk_Prob', 'Verdict']
display(budget_targets.head(10)[cols_to_show])

Current Season Median Minutes: 219.0

Total evaluated external budget players: 378



,Player,team_name,element_type,Cost_mil,minutes,Risk_Prob,Verdict
617,Rodrigo Bentancur,Spurs,MID,5.5,357,0.0,✅ Green: Durable Target
522,Kobbie Mainoo,Man Utd,MID,5.5,275,0.0,✅ Green: Durable Target
472,Rúben dos Santos Gato Alves Dias,Man City,DEF,5.5,360,0.0,✅ Green: Durable Target
467,Gianluigi Donnarumma,Man City,GKP,5.5,360,0.0,✅ Green: Durable Target
393,Julio Enciso,Ipswich Town,MID,5.5,340,0.0,✅ Green: Durable Target
473,Joško Gvardiol,Man City,DEF,5.7,344,0.0,✅ Green: Durable Target
380,Emersonn Correia da Silva,Ipswich Town,FWD,5.5,237,0.0,✅ Green: Durable Target
97,Keane Lewis-Potter,Brentford,MID,5.5,406,0.0,✅ Green: Durable Target
306,Josh King,Fulham,MID,5.5,346,0.0,✅ Green: Durable Target
281,Tyrique George,Everton,MID,5.5,298,0.0,✅ Green: Durable Target
